[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/3-3_RUL%EC%98%88%EC%B8%A12_%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D%ED%9A%8C%EA%B7%80.ipynb)

# 3-3 · RUL 예측 ② — 과거 고장 이력에서 머신러닝으로 배우기

**⏱ 100분** · 모듈 3: 열화 지표 → **③ RUL 예측** → 교체 시점 결정

## 🎯 이번 시간의 질문

> 과거 33개 필터의 고장 이력을 보고, "**센서가 이런 상태면 보통 몇 초 남았더라**"를 기계가 배우게 하면 어떨까요?

3-2의 추세 외삽은 "열화가 직선으로 진행하고, 모두 같은 기준선에서 고장 난다"는 **사람이 정한 가정**에 의존했고, 그 가정이 깨지면 크게 틀렸습니다.
이번에는 가정 대신 **데이터에서 직접 관계를 배우는** 방법을 씁니다. 2-3의 고장진단과 같은 지도학습인데, 정답이 상태(0/1/2)가 아니라 **숫자(남은 초)**라서 **회귀(regression)**라고 부릅니다.

| | 2-3 고장진단 (분류) | 3-3 RUL 예측 (회귀) |
|---|---|---|
| 정답 y | 누설 상태 0 / 1 / 2 | 잔여수명 RUL (초) |
| 모델 | `...Classifier` | `...Regressor` |
| 평가 | 정확도, 혼동행렬 | 평균 절대 오차 MAE (초) |
| 코드 패턴 | `fit` → `predict` | **똑같이** `fit` → `predict` |

**이번 시간에 익힐 패턴:** P5 회귀 모델 세 줄 · MAE 계산 · 예측 vs 실제 그래프 · 정답을 다듬는 방법(상한 자르기)

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/filter_runs.csv",
             "data/filter_windows.csv",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

## 1. 입력 특징 준비 (▶)

모델에 "지금 필터 상태"를 알려 줄 특징 여섯 개를 만듭니다. **모두 지금 시점까지의 데이터로만** 계산합니다(미래를 보면 안 됨).

| 특징 | 뜻 | 왜 필요한가 |
|---|---|---|
| `hi` | 전류 30초 이동평균 (3-1의 HI) | 지금 얼마나 막혔나 |
| `rise` | 실험 시작 때보다 HI가 오른 양 | 필터마다 처음 전류가 달라서, "얼마나 **변했나**"를 따로 봄 |
| `slope` | 최근 60초 동안 HI가 오른 양 | 지금 **빠르게** 나빠지는 중인가 |
| `time_s` | 지금까지 사용한 시간 | 오래 쓴 필터일수록 남은 시간이 적을 것 |
| `voltage_v` | 전압 | 전기 부하의 다른 측면 |
| `front_c` | 필터 앞쪽 온도 | 녹은 플라스틱 상태 |

In [ ]:
# ▶ 그대로 실행하세요
runs = pd.read_csv('data/filter_runs.csv')
windows = pd.read_csv('data/filter_windows.csv')
main = windows[windows['status'] == 'main'].copy()
g = main.groupby('run_id')
main['hi'] = g['current_a'].transform(lambda s: s.rolling(3, min_periods=1).mean())
main['rise'] = main['hi'] - main.groupby('run_id')['hi'].transform('first')
main['slope'] = main.groupby('run_id')['hi'].diff(6).fillna(0)     # 처음 60초는 변화 0으로 둠

cols = ['hi', 'rise', 'slope', 'time_s', 'voltage_v', 'front_c']
train = main[main['split'] == 'train']
test = main[main['split'] == 'test']
print('학습:', train['run_id'].nunique(), '개 필터,', len(train), '행 / 평가:', test['run_id'].nunique(), '개 필터,', len(test), '행')

⚠️ 학습용과 평가용은 **필터(실험) 단위**로 나눴습니다. 같은 필터의 앞부분이 학습에, 뒷부분이 평가에 들어가면 그 필터의 특성을 미리 본 셈이 되기 때문입니다(2-3의 "쌍둥이 사이클"과 같은 이유).

## 2. X와 y, 그리고 기준 모델

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
X_train = train[cols]
y_train = train['RUL_s']
X_test = test[cols]
y_test = test['RUL_s']
print(X_train.shape, X_test.shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**평가 지표 MAE(평균 절대 오차)** = `|예측 − 실제|`의 평균. "평균적으로 몇 초 틀리나"입니다. `sklearn`의 `mean_absolute_error`로 계산합니다.

먼저 3-2의 **기준 모델**("학습용 수명 중앙값 425초 − 지금 시간")의 MAE를 구해 둡니다. 새 모델은 이것보다 나아야 의미가 있습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
from sklearn.metrics import mean_absolute_error

life = runs[runs['split'] == 'train']['failure_s'].median()
pred_base = np.maximum(0, life - test['time_s'])
print('기준 모델 MAE:', round(mean_absolute_error(y_test, pred_base)), '초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


## 3. 선형회귀와 랜덤 포레스트 회귀

**P5 · 모델 세 줄** — 2-3과 모델 이름만 다릅니다.
- **선형회귀(LinearRegression)**: `RUL = 절편 + 계수1×hi + 계수2×rise + …` 형태의 식을 찾습니다. 음수 예측은 0으로 자릅니다.
- **랜덤 포레스트 회귀(RandomForestRegressor)**: 결정나무 100그루의 예측을 **평균**합니다(분류는 투표, 회귀는 평균). `min_samples_leaf=20`은 "나무의 맨 끝 칸에 최소 20개 행이 모이게" 해서 학습 데이터를 너무 세세하게 외우지 않게 합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = np.maximum(0, lr.predict(X_test))

rf = RandomForestRegressor(min_samples_leaf=20, random_state=0)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print('선형회귀 MAE:', round(mean_absolute_error(y_test, pred_lr)), '초')
print('랜덤 포레스트 MAE:', round(mean_absolute_error(y_test, pred_rf)), '초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 기준 399초, 선형회귀 284초, 랜덤 포레스트 282초.

💬 **결과 해설** — 센서를 쓰는 두 모델 모두 기준보다 **약 115초 덜 틀립니다.** 하지만 평균 280초 오차는 여전히 큽니다. 어디서 많이 틀릴까요?
3-2처럼 **실제 RUL 구간별**로 나눠 봅니다(함수 제공).

In [ ]:
# ▶ 그대로 실행하세요
def by_range(pred):
    """실제 RUL 구간별 평균 절대 오차(초)."""
    error = (pd.Series(pred, index=y_test.index) - y_test).abs()
    zone = pd.cut(y_test, [0, 100, 200, 300, 5000], right=False,
                  labels=['0~100초', '100~200초', '200~300초', '300초 이상'])
    return error.groupby(zone).mean().round()

pd.DataFrame({'기준 모델': by_range(pred_base), '선형회귀': by_range(pred_lr), '랜덤 포레스트': by_range(pred_rf)})

💬 **결과 해설** — 오차의 대부분은 **RUL이 300초 이상 남은(아직 건강한) 구간**에서 나옵니다(약 400초). 필터가 아직 막히기 전이라 센서에 단서가 거의 없는데, 그 필터가 앞으로 300초를 버틸지 1,000초를 버틸지 맞히라는 것은 무리입니다.
고장이 가까운 구간을 보면, **100~300초**에서는 랜덤 포레스트가 **96~106초**로 가장 낫지만, **0~100초**에서는 기준 모델(95초)보다도 못합니다(114초). (3-2 추세 외삽 기본 설정은 148~319초)

예측을 필터별로 그려 확인합니다.

In [ ]:
# ▶ 그대로 실행하세요
fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
for ax, (rid, run) in zip(axes.flat, test.groupby('run_id')):
    ax.plot(run['time_s'], run['RUL_s'], color='black', label='실제')
    ax.plot(run['time_s'], pred_rf[test['run_id'] == rid], label='랜덤 포레스트')
    ax.set_title(f'{rid}번 (F = {run["failure_s"].iloc[0]}초)')
    ax.set_xlabel('경과 시간 (초)')
axes[0, 0].set_ylabel('RUL (초)'); axes[1, 0].set_ylabel('RUL (초)')
axes[0, 0].legend()
fig.tight_layout()
plt.show()

💬 **결과 해설** — 수명이 긴 필터(4번 1,064초, 8번 1,596초)는 초반에 예측이 **300~600초 근처에 머물러** 실제보다 한참 작습니다. 모델이 학습한 필터 대부분의 수명이 그보다 짧았기 때문입니다.
반대로 고장 직전에는 예측이 실제보다 **위에** 있는 경우가 많습니다. 이 점은 4절에서 자세히 봅니다.

## 4. 핵심 개선 — "먼 미래는 맞히려 하지 않는다" (RUL 상한 자르기)

3절의 결론: 건강한 구간의 RUL은 원래 맞힐 수 없고, 모델이 그것까지 맞히려다 **정작 중요한 고장 직전 구간의 학습이 흐려집니다.**
그래서 실무에서는 정답 RUL을 **상한 300초로 잘라서** 가르칩니다. 300초보다 많이 남았으면 모두 "300초 (= 아직 충분히 남음)"으로 봅니다. 정비 결정에 필요한 정보는 "**300초 안쪽에서 얼마나 남았나**"이기 때문입니다.

```
원래 정답:   1000, 990, …, 310, 300, 290, …, 10, 0
자른 정답:    300, 300, …, 300, 300, 290, …, 10, 0
```

`y_train.clip(upper=300)` = 300보다 큰 값을 300으로 자르기.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
y_train_cap = y_train.clip(upper=300)

rf_cap = RandomForestRegressor(min_samples_leaf=20, random_state=0)
rf_cap.fit(X_train, y_train_cap)
pred_cap = rf_cap.predict(X_test)

near = y_test <= 300                     # 평가: 실제 RUL이 300초 이하인 구간 (정비 결정 구간)
print('랜덤 포레스트 (원래 정답):', round(mean_absolute_error(y_test[near], pred_rf[near])), '초')
print('랜덤 포레스트 (상한 300초):', round(mean_absolute_error(y_test[near], pred_cap[near])), '초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 고장 300초 이내 구간의 MAE: 106초 → **75초**.

💬 **결과 해설** — **모델도 특징도 그대로이고 정답만 다듬었는데** 결정 구간의 오차가 약 30% 줄었습니다. 모델이 "건강한지 / 얼마나 임박했는지"에 집중하게 되었기 때문입니다.
이 방법(구간 선형 RUL, piecewise-linear RUL)은 항공기 엔진 RUL 예측 등에서 표준처럼 쓰입니다. 대신 이 모델은 "300초 이상 남음"까지만 말할 수 있다는 한계를 받아들인 것입니다.

### ⚠️ 고장 직전, 모델은 낙관적인가 비관적인가?

정비에서 가장 위험한 실수는 "**아직 남았다**고 했는데 고장 나는 것"(과대 예측)입니다. 실제 RUL이 100초 미만인 시점에서 예측이 실제보다 큰 비율을 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
end = y_test < 100
over = pred_cap[end] > y_test[end]
print('고장 100초 전 구간:', end.sum(), '개 시점')
print('실제보다 길게(낙관적으로) 예측한 비율:', round(over.mean() * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 고장 100초 전 구간에서 **94%**를 실제보다 길게 예측했습니다. 모델이 체계적으로 **낙관적**입니다.
학습용 필터 중에 높은 전류에서 오래 버틴 경우(3-2의 18번 등)가 많아서 "이 정도 전류면 아직 좀 남았다"고 배웠기 때문입니다.
➡️ 그래서 이 예측으로 교체 시점을 정할 때는 **안전 여유**를 둬야 합니다. 예) "예측 RUL이 150초 이하가 되면 교체" (3-4에서 결정).

## 5. 모델은 무엇을 보고 판단했나

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
importance = pd.Series(rf_cap.feature_importances_, index=cols).sort_values()
plt.barh(importance.index, importance.values)
plt.xlabel('특징 중요도')
plt.title('RUL 예측(상한 300초)에 쓰인 특징')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 시작 대비 상승량 `rise`(0.29)와 현재 HI(0.26)가 가장 중요하고, 사용 시간 `time_s`(0.21), 전압(0.20)이 뒤를 잇습니다. "**얼마나 막혔나 + 얼마나 오래 썼나**"를 함께 본다는 뜻으로, 물리적으로 말이 됩니다. 최근 상승 속도 `slope`는 잡음이 커서 거의 쓰이지 않았습니다.

---
## 6. 실습 과제 (35분)

### 📝 과제 1 · 특징이 많을수록 좋을까?
상한 300초 모델에서 입력을 `['hi']` 하나만, `['hi', 'rise', 'slope']` 세 개, `cols` 여섯 개로 바꿔 결정 구간(RUL ≤ 300) MAE를 비교하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for c in [['hi'], ['hi', 'rise', 'slope'], cols]:  모델을 만들어 train[c], y_train_cap으로 fit, test[c]로 predict


✏️ **나의 답:**

### 📝 과제 2 · 상한을 얼마로 자를까?
상한을 **200, 300, 400, 600초, 자르지 않음**으로 바꿔, 실제 RUL이 **200초 미만**인 구간의 MAE를 비교하세요. (상한이 200초 이상이면 이 구간의 정답은 바뀌지 않으므로 공정한 비교입니다.)

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for cap in [200, 300, 400, 600, None]: y = y_train if cap is None else y_train.clip(upper=cap)


✏️ **나의 답:** 상한을 낮출수록 고장 직전 오차는? 그럼 상한을 아주 낮게(예: 50초) 하면 좋을까요?

### 📝 과제 3 · 나무를 얼마나 세세하게?
`min_samples_leaf`를 **1, 5, 20, 50**으로 바꿔 **학습 데이터의 MAE**와 **평가 데이터의 MAE**(모두 RUL ≤ 300 구간, 상한 300 모델)를 비교하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 학습 MAE는 m.predict(X_train)과 y_train_cap을 비교합니다. 학습용의 결정 구간은 train['RUL_s'] <= 300


✏️ **나의 답:**

### 📝 과제 4 · 추세 외삽 vs 머신러닝 — 언제 무엇을?
3-2(추세 외삽)와 3-3(랜덤 포레스트)의 결과를 비교하는 표를 만들고, 두 방법의 장단점을 적으세요. 참고 수치:
- 추세 외삽 최선 설정(N=20, 기준선 1.0 A): 0~100초 51초, 100~200초 126초, 200~300초 225초
- 랜덤 포레스트(상한 300초): 아래 셀로 구간별 오차를 구하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: by_range(pred_cap)


✏️ **나의 답:**

| | 추세 외삽 | 랜덤 포레스트 |
|---|---|---|
| 0~100초 오차 | 51초 | |
| 100~300초 오차 | | |
| 필요한 데이터 | | |
| 설명하기 쉬운가 | | |
| 잘 맞는 상황 | | |

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| RUL 회귀 | 정답이 숫자(초)인 지도학습. 코드 패턴은 분류와 똑같이 `fit` → `predict` |
| MAE | 평균적으로 몇 초 틀리나. **실제 RUL 구간별로** 나눠 봐야 의미가 보인다 |
| 먼 미래의 한계 | 건강한 구간의 RUL은 센서로 알 수 없다 → 오차의 대부분 |
| 상한 자르기 | 정답을 300초로 잘라 결정 구간에 집중 → 결정 구간 오차 106 → 75초 |
| 낙관 편향 | 고장 직전 94%를 길게 예측 → 결정할 때 **안전 여유** 필요 |

다음: **3-4 · RUL로 교체 시점 결정하기 — 시간 기준 vs 상태 기준 vs 예측 기준**